In [37]:
import pandas as pd 
import joblib

In [2]:
import os
os.listdir("../dataset")

['.ipynb_checkpoints', 'hindi_songs_dataset_50k-1.csv']

In [3]:
df = pd.read_csv("../dataset/hindi_songs_dataset_50k-1.csv")
df.head()

,song_id,song_title,singer,music_director,lyricist,album_movie,release_year,genre,mood,duration
0,1,Barish Kehna Hai,Amit Trivedi,Vishal-Shekhar,Kumaar,Sapne 2,2010,EDM,Happy,2:36
1,2,Dhadkan Karta Hoon,Nikhita Gandhi,Laxmikant-Pyarelal,Manoj Muntashir,Khushi Ki Raat,1975,Classical / Semi-Classical,Happy,3:29
2,3,Jaan Sun Le,Kumar Sanu,Himesh Reshammiya,Javed Akhtar,Baazigar,2018,Item Song,Devotional,5:28
3,4,Tera Sapna,Dhvani Bhanushali,R.D. Burman,Shakeel Badayuni,Ishq 2,1986,Sad / Heartbreak,Bittersweet,4:27
4,5,Tera Manzil,Dhvani Bhanushali,Nadeem-Shravan,Anand Bakshi,Baazigar Wala Pyaar,1984,Patriotic,Bittersweet,2:50


In [4]:
df.shape

(50000, 10)

In [5]:
df.isnull().sum()

song_id           0
song_title        0
singer            0
music_director    0
lyricist          0
album_movie       0
release_year      0
genre             0
mood              0
duration          0
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
# Check unique vlaues in genre mood and singer 
print(df["genre"].unique())
print(df["mood"].unique())
print(df["singer"].nunique())

<ArrowStringArray>
[                       'EDM', 'Classical / Semi-Classical',
                  'Item Song',           'Sad / Heartbreak',
                  'Patriotic',              'Party / Dance',
                    'Lyrical',             'Retro / Melody',
                   'Romantic',                    'Lullaby',
          'Wedding / Sangeet',                       'Rock',
                        'Pop',                      'Indie',
              'Hip-Hop / Rap',        'Devotional / Bhajan',
                       'Sufi',                       'Folk',
                     'Ghazal',                    'Qawwali']
Length: 20, dtype: str
<ArrowStringArray>
[      'Happy',  'Devotional', 'Bittersweet',   'Energetic',     'Playful',
 'Melancholic',    'Romantic',    'Soothing',   'Uplifting',   'Nostalgic',
 'Celebratory',     'Intense']
Length: 12, dtype: str
413


In [8]:
df["mood"].value_counts()

mood
Bittersweet    4248
Intense        4248
Melancholic    4215
Uplifting      4213
Nostalgic      4177
Energetic      4169
Celebratory    4154
Devotional     4134
Happy          4133
Soothing       4133
Romantic       4094
Playful        4082
Name: count, dtype: int64

In [9]:
df["genre"].value_counts()

genre
Rock                          2592
Romantic                      2576
Patriotic                     2537
Wedding / Sangeet             2530
Indie                         2529
Qawwali                       2527
Party / Dance                 2525
Pop                           2525
Sufi                          2519
Lyrical                       2492
Devotional / Bhajan           2492
Sad / Heartbreak              2488
Hip-Hop / Rap                 2486
Folk                          2473
Retro / Melody                2472
EDM                           2469
Item Song                     2455
Classical / Semi-Classical    2448
Ghazal                        2443
Lullaby                       2422
Name: count, dtype: int64

In [10]:
df["song_title"].duplicated().sum()

np.int64(46108)

In [11]:
df["song_title"].value_counts().head()

song_title
Dhadkan Mera      43
Milan Ho          42
Armaan Ho Gaya    42
Yaad Se Pyaar     41
Khwaab Mera       41
Name: count, dtype: int64

In [12]:
df.duplicated(subset=["song_title", "singer"]).sum()

np.int64(7478)

In [13]:
print(df["release_year"].min())
print(df["release_year"].max())

1970
2026


In [14]:
## convert minutes into seconds

def duration_to_sec(time):
    minutes, seconds = time.split(":")
    return int(minutes) * 60 + int(seconds)

In [15]:
# make a duration second column
df["duration_seconds"] = df["duration"].apply(duration_to_sec)

In [16]:
df.head()

,song_id,song_title,singer,music_director,lyricist,album_movie,release_year,genre,mood,duration,duration_seconds
0,1,Barish Kehna Hai,Amit Trivedi,Vishal-Shekhar,Kumaar,Sapne 2,2010,EDM,Happy,2:36,156
1,2,Dhadkan Karta Hoon,Nikhita Gandhi,Laxmikant-Pyarelal,Manoj Muntashir,Khushi Ki Raat,1975,Classical / Semi-Classical,Happy,3:29,209
2,3,Jaan Sun Le,Kumar Sanu,Himesh Reshammiya,Javed Akhtar,Baazigar,2018,Item Song,Devotional,5:28,328
3,4,Tera Sapna,Dhvani Bhanushali,R.D. Burman,Shakeel Badayuni,Ishq 2,1986,Sad / Heartbreak,Bittersweet,4:27,267
4,5,Tera Manzil,Dhvani Bhanushali,Nadeem-Shravan,Anand Bakshi,Baazigar Wala Pyaar,1984,Patriotic,Bittersweet,2:50,170


In [17]:
## Make a combine feature columns so that model can analyse the recommendation
df["combined_features"] = df[["singer", "music_director", "lyricist","album_movie", "genre", "mood"]
    ].astype(str).agg(" ".join, axis=1)

In [18]:
df.head()

,song_id,song_title,singer,music_director,lyricist,album_movie,release_year,genre,mood,duration,duration_seconds,combined_features
0,1,Barish Kehna Hai,Amit Trivedi,Vishal-Shekhar,Kumaar,Sapne 2,2010,EDM,Happy,2:36,156,Amit Trivedi Vishal-Shekhar Kumaar Sapne 2 EDM...
1,2,Dhadkan Karta Hoon,Nikhita Gandhi,Laxmikant-Pyarelal,Manoj Muntashir,Khushi Ki Raat,1975,Classical / Semi-Classical,Happy,3:29,209,Nikhita Gandhi Laxmikant-Pyarelal Manoj Muntas...
2,3,Jaan Sun Le,Kumar Sanu,Himesh Reshammiya,Javed Akhtar,Baazigar,2018,Item Song,Devotional,5:28,328,Kumar Sanu Himesh Reshammiya Javed Akhtar Baaz...
3,4,Tera Sapna,Dhvani Bhanushali,R.D. Burman,Shakeel Badayuni,Ishq 2,1986,Sad / Heartbreak,Bittersweet,4:27,267,Dhvani Bhanushali R.D. Burman Shakeel Badayuni...
4,5,Tera Manzil,Dhvani Bhanushali,Nadeem-Shravan,Anand Bakshi,Baazigar Wala Pyaar,1984,Patriotic,Bittersweet,2:50,170,Dhvani Bhanushali Nadeem-Shravan Anand Bakshi ...


In [19]:
# Make song title in lower case
df["song_lower_title"] = df["song_title"].str.lower()

In [20]:
## Use TF-IDF numerical verctors 
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()

In [21]:
tfidf_matrix = tfidf.fit_transform(df["combined_features"])
tfidf_matrix.shape

(50000, 204)

In [22]:
# Calculate similarity between song vectors woth cosine similarity

from sklearn.metrics.pairwise import cosine_similarity

In [23]:
similarity_matrix = cosine_similarity(tfidf_matrix)

In [24]:
similarity_matrix.shape

(50000, 50000)

In [25]:
# find recommend songs based on  1 input song..
def recommend(songs):
    song_index = df[df["song_title"] == songs].index[0]
    similarity_score = similarity_matrix[song_index]
    song_list = sorted(list(enumerate(similarity_score)), reverse = True,  key = lambda x : x[1])

    top_songs = song_list[1:11]
    for i in top_songs:
        print(df.iloc[i[0]]["song_title"])

In [26]:
recommend("Barish Kehna Hai")

Tera Zaroori Hai
Tera Kehna Hai
Khushi Chahiye
Tanhai Judaai
Vaada Sawan
Vaada Kehna Hai
Khushi Wali Raat
Kasam Bulaye
Wafa Se Pyaar
Nasha Ho Gaya


In [27]:
# now enter 5 input songs and get output related to these songs 
songs = ["Barish Kehna Hai","Tera Sapna","Tera Manzil","Jaan Sun Le", "Dhadkan Karta Hoon"]

In [28]:
#  find index 
song_indexes = [
    df[df["song_title"] == song].index[0]
    for song in songs
]

In [29]:
# find similarity 
input_similarities = similarity_matrix[song_indexes]

In [30]:
# find avg similarity
combined_scores = input_similarities.mean(axis=0)

In [31]:
# remove imput songs
combined_scores[song_indexes] = -1

In [32]:
# use enumerate and sort the songs list
songs_list = list(enumerate(combined_scores))

songs_list = sorted(songs_list, reverse=True, key=lambda x: x[1])

In [33]:
# Get top 10 song
top_songs = songs_list[:10]

In [34]:
# show the songs list
for i in top_songs:
    print(df.iloc[i[0]]["song_title"])

Yaad Nazar
Toofan Milta Hai
Rang Sun Le
Pyaar Ki Kahani
Zindagi Ki Chah
Lehar Ki Dastaan
Lehar Nasha
Dil Ho
Wafa Bin Tere
Guzarish Wala Ishq


In [35]:
# Make the function which get 5 inputs and then recommend 10 relevent list of songs

def recommend_songs(songs, n=10):
    songs = [song.lower() for song in songs]

    song_indexes = []

    for song in songs:

        matches = df[df["song_lower_title"] == song]

        if matches.empty:
            print(f"Song not found: {song}")
            return

        song_indexes.append(matches.index[0])
        
    input_similarities = similarity_matrix[song_indexes]

    # find average similarity score
    combine_scores = input_similarities.mean(axis=0)

    # Exclude input songs
    combine_scores[song_indexes] = -1

    # make index-score pairs
    songs_list = list(enumerate(combine_scores))

    # Sort similarity score
    songs_list = sorted(songs_list, reverse=True, key=lambda x: x[1])

    # get list of songs which we want to recommend
    top_songs = songs_list[:n]

    for i in top_songs:
        print(df.iloc[i[0]]["song_title"])

In [36]:
songs = [
    "Barish Kehna Hai",
    "tera sapna",
    "Tera Manzil",
    "Dil ho",
    "Dhadkan Karta Hoon"
]

recommend_songs(songs)

Mehfil Ke Sang
Dooriyan Sun Le
Sapna Dhadkan
Judaai Pyaar
Manzil Chala Gaya
Sitara Mohabbat
Uska Ke Sang
Dhadkan Tera
Silsila Hamdard
Rang Dhadkan


In [41]:
# Save tfidf , tfidf matrix and dataset 
joblib.dump(tfidf, "../model/tfidf_vectorizer.joblib")
joblib.dump(tfidf_matrix, "../model/tfidf_matrix.joblib")
joblib.dump(df, "../model/songs_data.joblib")

['../model/songs_data.joblib']